# 02 ground truth preparation

Code-only reproduction copy. Outputs and execution counts have been cleared. Configure the lab settings and provide the documented input data before running. Original research evidence remains outside this folder.


In [ ]:
# Keep all inputs and outputs inside this reproduction copy.
from pathlib import Path
import sys
import re

_here = Path.cwd().resolve()
PUBLICATION_ROOT = next((p for p in (_here, _here.parent)
    if (p / '.publication-root').is_file()), None)
if PUBLICATION_ROOT is None:
    raise FileNotFoundError('Start the kernel in this repository root or its notebooks directory.')
if str(PUBLICATION_ROOT) not in sys.path:
    sys.path.insert(0, str(PUBLICATION_ROOT))
from src.lab_config import load_lab_config
LAB_CONFIG = load_lab_config()


In [ ]:
from pathlib import Path
import json
import pandas as pd

# Locate the M05 result files
DATA_DIR = Path.cwd().parent / "data" / "ground_truth"
M05_DIR = DATA_DIR / "M05Results"

m05_files = sorted(M05_DIR.glob("*.json"))

print("Total M05 result files:", len(m05_files))

# Load the first result file
with open(m05_files[0], "r", encoding="utf-8-sig") as file:
    m05_result = json.load(file)

# Display the available fields
print("\nFields in the M05 result:")
print(list(m05_result.keys()))

# Display selected execution evidence
for key in [
    "SessionID",
    "Status",
    "FilesRequested",
    "FilesTransferred",
    "TransferPerformed"
]:
    print(f"{key}: {m05_result.get(key)}")

In [ ]:
# Read all M05 result files
m05_results = []

for file_path in m05_files:
    with open(file_path, "r", encoding="utf-8-sig") as file:
        result = json.load(file)

    m05_results.append({
        "SessionID": result.get("SessionID"),
        "Status": result.get("Status"),
        "FilesRequested": result.get("FilesRequested"),
        "FilesTransferred": result.get("FilesTransferred"),
        "TransferPerformed": result.get("TransferPerformed"),
        "StartedUTC": result.get("StartedUTC"),
        "EndedUTC": result.get("EndedUTC")
    })

# Convert the list of dictionaries into a DataFrame
m05_df = pd.DataFrame(m05_results)

print("Total result records:", len(m05_df))

print("\nM05 worker statuses:")
print(m05_df["Status"].value_counts(dropna=False))

display(m05_df.head())

In [ ]:
# Load the original simulation schedule
schedule = pd.read_csv(
    DATA_DIR / "dataset_schedule_v3.csv"
)

# Select only scheduled M05 sessions
scheduled_m05 = schedule[
    schedule["ScenarioID"] == "M05"
].copy()

print("Scheduled M05 sessions:", len(scheduled_m05))

# Match worker results to scheduled M05 sessions
matched_m05 = scheduled_m05.merge(
    m05_df,
    on="SessionID",
    how="left",
    validate="one_to_one",
    indicator=True
)

print("\nMatching results:")
print(matched_m05["_merge"].value_counts())

print("\nScheduled M05 worker statuses:")
print(matched_m05["Status"].value_counts(dropna=False))

In [ ]:
# Check whether each scheduled M05 session successfully transferred files
matched_m05["TransferVerified"] = (
    (matched_m05["Status"] == "Completed") &
    (matched_m05["FilesRequested"] > 0) &
    (matched_m05["FilesTransferred"] == matched_m05["FilesRequested"]) &
    (matched_m05["TransferPerformed"] == True)
)

print("Transfer verification results:")
print(matched_m05["TransferVerified"].value_counts())

# Examine the session that did not pass verification
print("\nSessions without verified worker transfer:")
display(
    matched_m05.loc[
        ~matched_m05["TransferVerified"],
        [
            "SessionID",
            "Status",
            "FilesRequested",
            "FilesTransferred",
            "TransferPerformed"
        ]
    ]
)

In [ ]:
# Load the execution ledger
ledger = pd.read_csv(DATA_DIR / "execution_ledger.csv")

# Get the final execution record for each session
final_ledger = ledger.drop_duplicates(
    subset="SessionID",
    keep="last"
)

# Combine schedule with execution outcomes
ground_truth = schedule.merge(
    final_ledger[["SessionID", "Status"]],
    on="SessionID",
    how="left",
    validate="one_to_one"
)

# Add M05 worker verification
m05_evidence = matched_m05[
    ["SessionID", "TransferVerified"]
]

ground_truth = ground_truth.merge(
    m05_evidence,
    on="SessionID",
    how="left",
    validate="one_to_one"
)

# Assign provisional labels only to completed sessions
ground_truth["GroundTruthLabel"] = (
    ground_truth["IntendedLabel"]
    .where(ground_truth["Status"] == "Completed", pd.NA)
    .astype("Int64")
)

# Check agreement between M05 worker verification and execution status
m05_check = ground_truth[
    ground_truth["ScenarioID"] == "M05"
]

display(
    pd.crosstab(
        m05_check["Status"],
        m05_check["TransferVerified"],
        dropna=False
    )
)

print("Total ground-truth records:", len(ground_truth))

In [ ]:
print("Ground-truth label distribution:")
print(ground_truth["GroundTruthLabel"].value_counts(dropna=False))

print("\nMissing execution statuses:")
print(ground_truth["Status"].isna().sum())

print("\nDuplicate SessionIDs:")
print(ground_truth["SessionID"].duplicated().sum())

# Confirm that successful M05 sessions have worker transfer evidence
assert len(ground_truth) == 1200
assert ground_truth["SessionID"].is_unique
assert ground_truth["Status"].notna().all()
assert (m05_check["Status"].eq("Completed") == m05_check["TransferVerified"]).all()

print("\nBasic validation checks passed.")

In [ ]:
# Compare the original schedule with the final execution ledger
consistency_check = schedule.merge(
    final_ledger[
        ["SessionID", "Username", "ScenarioID", "ScheduledUTC"]
    ],
    on="SessionID",
    how="left",
    suffixes=("_schedule", "_ledger"),
    validate="one_to_one"
)

# Compare usernames
username_mismatch = (
    consistency_check["Username_schedule"] !=
    consistency_check["Username_ledger"]
)

# Compare scenario identifiers
scenario_mismatch = (
    consistency_check["ScenarioID_schedule"] !=
    consistency_check["ScenarioID_ledger"]
)

print("Username mismatches:", username_mismatch.sum())
print("Scenario mismatches:", scenario_mismatch.sum())

# Show any inconsistent records
display(
    consistency_check.loc[
        username_mismatch | scenario_mismatch,
        [
            "SessionID",
            "Username_schedule",
            "Username_ledger",
            "ScenarioID_schedule",
            "ScenarioID_ledger"
        ]
    ]
)

In [ ]:
# Create a separate folder for datasets generated in Jupyter
OUTPUT_DIR = Path.cwd().parent / "data" / "notebook_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save our ground-truth dataset
output_file = OUTPUT_DIR / "session_ground_truth.csv"

ground_truth.to_csv(output_file, index=False)

print(f"Ground-truth dataset saved to: {output_file}")
print(f"Total sessions: {len(ground_truth)}")